# Part IV: Connections - Code Examples

This notebook covers Chapters 13-15:
- **Chapter 13**: Quantum vs Classical Complexity — why $C_q < C_\mu$
- **Chapter 14**: Error Correction — protecting coherence
- **Chapter 15**: Open Questions — what we don't know

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## Chapter 13: Quantum vs Classical Complexity

The key result: for almost all processes, $C_q < C_\mu$.

### Building Q-Machines from ε-Machines

In [4]:
def von_neumann_entropy(rho):
    """S(ρ) = -Tr(ρ log₂ ρ)"""
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-12]
    if len(eigenvalues) == 0:
        return 0.0
    return -np.sum(eigenvalues * np.log2(eigenvalues))


def shannon_entropy(probs):
    """H(p) = -Σ p log₂ p"""
    probs = np.array(probs)
    probs = probs[probs > 1e-12]
    return -np.sum(probs * np.log2(probs))


def signal_state_simple(transition_probs):
    """
    Build signal state for simple 2-state, 2-symbol case.
    transition_probs[x] = probability of emitting symbol x
    (all transitions go to the same next state in perturbed coin)
    """
    return np.array([np.sqrt(p) for p in transition_probs], dtype=complex)


def quantum_complexity_simple(s_states, pi):
    """
    Compute C_q from list of signal states and stationary distribution.
    """
    dim = len(s_states[0])
    rho = np.zeros((dim, dim), dtype=complex)
    for j, pj in enumerate(pi):
        s = s_states[j]
        rho += pj * np.outer(s, s.conj())

    C_q = von_neumann_entropy(rho.real)
    return C_q, rho


print("=== Quantum Complexity Infrastructure ===")

=== Quantum Complexity Infrastructure ===


In [5]:
# Perturbed Coin: the canonical example
print("=== Perturbed Coin: Complete Analysis ===\n")


def perturbed_coin_analysis(p):
    """
    Analyze perturbed coin for parameter p.

    The perturbed coin has two causal states A and B:
    - State A: emit 0 with prob (1-p), emit 1 with prob p, then go to B
    - State B: emit 0 with prob p, emit 1 with prob (1-p), then go to A

    Signal states encode the emission probabilities:
    |s_A⟩ = √(1-p)|0⟩ + √p|1⟩
    |s_B⟩ = √p|0⟩ + √(1-p)|1⟩
    """
    # Signal states (in emission basis)
    s_A = np.array([np.sqrt(1 - p), np.sqrt(p)])
    s_B = np.array([np.sqrt(p), np.sqrt(1 - p)])

    # Classical complexity
    pi = np.array([0.5, 0.5])
    C_mu = shannon_entropy(pi)

    # Quantum complexity
    C_q, rho = quantum_complexity_simple([s_A, s_B], pi)

    # Overlap
    overlap = np.abs(np.dot(s_A, s_B))

    return {
        "p": p,
        "C_mu": C_mu,
        "C_q": C_q,
        "advantage": C_mu - C_q,
        "overlap": overlap,
        "s_A": s_A,
        "s_B": s_B,
        "rho": rho,
    }


# Analyze for various p values
print(f"{'p':<8} {'C_μ':<10} {'C_q':<10} {'Advantage':<12} {'Overlap':<10}")
print("-" * 50)

for p in [0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.01]:
    result = perturbed_coin_analysis(p)
    print(
        f"{p:<8.2f} {result['C_mu']:<10.3f} {result['C_q']:<10.3f} "
        f"{result['advantage']:<12.3f} {result['overlap']:<10.3f}"
    )

=== Perturbed Coin: Complete Analysis ===

p        C_μ        C_q        Advantage    Overlap   
--------------------------------------------------
0.50     1.000      -0.000     1.000        1.000     
0.40     1.000      0.081      0.919        0.980     
0.30     1.000      0.250      0.750        0.917     
0.20     1.000      0.469      0.531        0.800     
0.10     1.000      0.722      0.278        0.600     
0.05     1.000      0.858      0.142        0.436     
0.01     1.000      0.971      0.029        0.199     


In [6]:
# Visualize the density matrix structure
print("=== Density Matrix at p = 0.3 ===\n")

result = perturbed_coin_analysis(0.3)
rho = result["rho"]

print("Signal states (in symbol basis |0⟩, |1⟩):")
print(f"  |s_A⟩ = {result['s_A']}")
print(f"  |s_B⟩ = {result['s_B']}")
print(f"  Overlap ⟨s_A|s_B⟩ = {result['overlap']:.4f}")

print(f"\nDensity matrix ρ = ½|s_A⟩⟨s_A| + ½|s_B⟩⟨s_B| (2×2):")
print(rho.real)

print(f"\nEigenvalues: {np.linalg.eigvalsh(rho.real)}")
print(f"Trace: {np.trace(rho.real):.4f} (should be 1)")
print(f"Von Neumann entropy: {result['C_q']:.4f} bits")

print("\n→ Off-diagonal elements (", rho[0, 1].real, ") show quantum coherence")
print("→ Eigenvalues [0.042, 0.958] are concentrated → low entropy")

=== Density Matrix at p = 0.3 ===

Signal states (in symbol basis |0⟩, |1⟩):
  |s_A⟩ = [0.8367 0.5477]
  |s_B⟩ = [0.5477 0.8367]
  Overlap ⟨s_A|s_B⟩ = 0.9165

Density matrix ρ = ½|s_A⟩⟨s_A| + ½|s_B⟩⟨s_B| (2×2):
[[0.5    0.4583]
 [0.4583 0.5   ]]

Eigenvalues: [0.0417 0.9583]
Trace: 1.0000 (should be 1)
Von Neumann entropy: 0.2502 bits

→ Off-diagonal elements ( 0.458257569495584 ) show quantum coherence
→ Eigenvalues [0.042, 0.958] are concentrated → low entropy


## Chapter 14: Error Correction

Three-qubit bit-flip code: the simplest quantum error correction.

In [7]:
# Three-qubit bit-flip code
print("=== Three-Qubit Bit-Flip Code ===\n")


def ket(bits):
    """Create 3-qubit basis state from bit string."""
    idx = int(bits, 2)
    state = np.zeros(8, dtype=complex)
    state[idx] = 1
    return state


# Logical states
ket_0L = ket("000")  # |0_L⟩ = |000⟩
ket_1L = ket("111")  # |1_L⟩ = |111⟩

# Superposition: α|0_L⟩ + β|1_L⟩
alpha, beta = 1 / np.sqrt(2), 1 / np.sqrt(2)
psi_L = alpha * ket_0L + beta * ket_1L

print("Logical encoding:")
print(f"  |0_L⟩ = |000⟩")
print(f"  |1_L⟩ = |111⟩")
print(f"\nEncoded state |ψ_L⟩ = α|0_L⟩ + β|1_L⟩ (α = β = 1/√2)")
print(f"  = (|000⟩ + |111⟩)/√2")

=== Three-Qubit Bit-Flip Code ===

Logical encoding:
  |0_L⟩ = |000⟩
  |1_L⟩ = |111⟩

Encoded state |ψ_L⟩ = α|0_L⟩ + β|1_L⟩ (α = β = 1/√2)
  = (|000⟩ + |111⟩)/√2


In [8]:
# Syndrome measurement operators
print("=== Syndrome Measurement ===\n")

# Build operators
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
I = np.eye(2, dtype=complex)

# X errors on each qubit
X_1 = np.kron(np.kron(X, I), I)
X_2 = np.kron(np.kron(I, X), I)
X_3 = np.kron(np.kron(I, I), X)

# Syndrome operators: parity checks
Z1Z2 = np.kron(np.kron(Z, Z), I)
Z2Z3 = np.kron(np.kron(I, Z), Z)


def get_syndrome(state):
    """Measure syndrome without destroying superposition."""
    s1 = np.real(state.conj() @ Z1Z2 @ state)
    s2 = np.real(state.conj() @ Z2Z3 @ state)
    return (int(s1 < 0), int(s2 < 0))  # Convert eigenvalue to bit


def diagnose(syndrome):
    """Map syndrome to error location."""
    syndrome_table = {
        (0, 0): "No error",
        (1, 0): "Error on qubit 1",
        (1, 1): "Error on qubit 2",
        (0, 1): "Error on qubit 3",
    }
    return syndrome_table.get(syndrome, "Unknown")


# Test all single-qubit errors
print("Syndrome table:")
print(f"{'Error':<20} {'State after error':<35} {'Syndrome':<12} {'Diagnosis'}")
print("-" * 80)

for name, error in [("None", np.eye(8)), ("X_1", X_1), ("X_2", X_2), ("X_3", X_3)]:
    state_error = error @ psi_L
    syn = get_syndrome(state_error)
    diag = diagnose(syn)
    # Show which basis states have amplitude
    nonzero = [format(i, "03b") for i in range(8) if abs(state_error[i]) > 0.1]
    print(f"{name:<20} {str(nonzero):<35} {str(syn):<12} {diag}")

=== Syndrome Measurement ===

Syndrome table:
Error                State after error                   Syndrome     Diagnosis
--------------------------------------------------------------------------------
None                 ['000', '111']                      (0, 0)       No error
X_1                  ['011', '100']                      (1, 0)       Error on qubit 1
X_2                  ['010', '101']                      (1, 1)       Error on qubit 2
X_3                  ['001', '110']                      (0, 1)       Error on qubit 3


In [9]:
# Full error correction cycle
print("=== Full Error Correction Cycle ===\n")

# Apply error on qubit 2
print("1. Start with logical state |ψ_L⟩ = (|000⟩ + |111⟩)/√2")
print(f"   State: {psi_L}")

psi_error = X_2 @ psi_L
print(f"\n2. Error occurs: X_2 flips qubit 2")
print(f"   State: {psi_error}")
print(f"   (Now it's |010⟩ + |101⟩)/√2")

syn = get_syndrome(psi_error)
print(f"\n3. Measure syndrome: {syn}")
print(f"   Diagnosis: {diagnose(syn)}")

# Correct by applying X_2 again
psi_corrected = X_2 @ psi_error
print(f"\n4. Apply correction: X_2")
print(f"   State: {psi_corrected}")

# Verify
matches = np.allclose(psi_corrected, psi_L)
print(f"\n5. Verify: matches original? {matches} ✓")
print("\n→ We detected and corrected the error WITHOUT learning α or β!")

=== Full Error Correction Cycle ===

1. Start with logical state |ψ_L⟩ = (|000⟩ + |111⟩)/√2
   State: [0.7071+0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
 0.    +0.j 0.7071+0.j]

2. Error occurs: X_2 flips qubit 2
   State: [0.    +0.j 0.    +0.j 0.7071+0.j 0.    +0.j 0.    +0.j 0.7071+0.j
 0.    +0.j 0.    +0.j]
   (Now it's |010⟩ + |101⟩)/√2

3. Measure syndrome: (1, 1)
   Diagnosis: Error on qubit 2

4. Apply correction: X_2
   State: [0.7071+0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j 0.    +0.j
 0.    +0.j 0.7071+0.j]

5. Verify: matches original? True ✓

→ We detected and corrected the error WITHOUT learning α or β!


## Chapter 15: Open Questions

Exploring the frontiers of quantum computational mechanics.

In [10]:
# Question 1: What structural features predict large quantum advantage?
print("=== Structural Features → Quantum Advantage ===\n")


def analyze_simple_process(name, signal_states, pi):
    """Analyze a process given its signal states."""
    C_mu = shannon_entropy(pi)
    C_q, rho = quantum_complexity_simple(signal_states, pi)
    advantage = C_mu - C_q

    # Compute average pairwise overlap
    n = len(signal_states)
    if n > 1:
        overlaps = []
        for i in range(n):
            for j in range(i + 1, n):
                overlaps.append(np.abs(np.dot(signal_states[i], signal_states[j])))
        avg_overlap = np.mean(overlaps)
    else:
        avg_overlap = 0.0

    return {
        "name": name,
        "C_mu": C_mu,
        "C_q": C_q,
        "advantage": advantage,
        "overlap": avg_overlap,
    }


# Test various processes using signal state representation

# 1. IID process - single state, orthogonal to itself (no advantage)
s_iid = [np.array([np.sqrt(0.5), np.sqrt(0.5)])]
pi_iid = np.array([1.0])

# 2. Orthogonal states (like golden mean) - no overlap
s_orth = [np.array([1.0, 0.0]), np.array([0.0, 1.0])]
pi_orth = np.array([0.5, 0.5])

# 3. Perturbed coin p=0.3 - high overlap
p = 0.3
s_perturbed = [np.array([np.sqrt(1 - p), np.sqrt(p)]), np.array([np.sqrt(p), np.sqrt(1 - p)])]
pi_perturbed = np.array([0.5, 0.5])

# 4. Perturbed coin p=0.1 - moderate overlap
p = 0.1
s_perturbed2 = [np.array([np.sqrt(1 - p), np.sqrt(p)]), np.array([np.sqrt(p), np.sqrt(1 - p)])]

# 5. Three nearly identical states
angle = 0.1
s_three = [
    np.array([np.cos(0 * angle), np.sin(0 * angle)]),
    np.array([np.cos(1 * angle), np.sin(1 * angle)]),
    np.array([np.cos(2 * angle), np.sin(2 * angle)]),
]
pi_three = np.array([1 / 3, 1 / 3, 1 / 3])

processes = [
    ("IID (single state)", s_iid, pi_iid),
    ("Orthogonal states", s_orth, pi_orth),
    ("Perturbed coin (p=0.3)", s_perturbed, pi_perturbed),
    ("Perturbed coin (p=0.1)", s_perturbed2, pi_perturbed),
    ("Three similar states", s_three, pi_three),
]

print(f"{'Process':<25} {'C_μ':<8} {'C_q':<8} {'Adv':<8} {'Overlap'}")
print("-" * 60)

for name, s, pi in processes:
    result = analyze_simple_process(name, s, pi)
    print(
        f"{result['name']:<25} {result['C_mu']:<8.3f} {result['C_q']:<8.3f} "
        f"{result['advantage']:<8.3f} {result['overlap']:.3f}"
    )

print("\n→ Higher overlap between signal states → more quantum advantage")

=== Structural Features → Quantum Advantage ===

Process                   C_μ      C_q      Adv      Overlap
------------------------------------------------------------
IID (single state)        -0.000   -0.000   0.000    0.000
Orthogonal states         1.000    1.000    0.000    0.000
Perturbed coin (p=0.3)    1.000    0.250    0.750    0.917
Perturbed coin (p=0.1)    1.000    0.722    0.278    0.600
Three similar states      1.585    0.058    1.527    0.990

→ Higher overlap between signal states → more quantum advantage


In [11]:
# Question 2: How does decoherence affect quantum advantage?
print("=== Decoherence Trajectory ===\n")
print("As coherence decays (dephasing), C_q → C_μ\n")


def apply_dephasing(rho, gamma):
    """
    Apply dephasing channel with strength gamma.
    gamma=0: no dephasing (pure quantum)
    gamma=1: full dephasing (classical)
    """
    rho_dephased = rho.copy()
    n = rho.shape[0]
    for i in range(n):
        for j in range(n):
            if i != j:
                rho_dephased[i, j] *= 1 - gamma
    return rho_dephased


# Start with perturbed coin at p=0.3
result = perturbed_coin_analysis(0.3)
rho_pure = result["rho"]
C_mu = result["C_mu"]

print(f"{'Dephasing γ':<15} {'C(γ)':<10} {'Advantage':<12}")
print("-" * 40)

for gamma in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    rho_dephased = apply_dephasing(rho_pure, gamma)
    C_gamma = von_neumann_entropy(rho_dephased.real)
    advantage = C_mu - C_gamma
    marker = " ← pure quantum" if gamma == 0 else (" ← fully classical" if gamma == 1 else "")
    print(f"{gamma:<15.1f} {C_gamma:<10.4f} {advantage:<12.4f}{marker}")

print("\n→ Decoherence destroys off-diagonals → entropy increases → advantage vanishes")

=== Decoherence Trajectory ===

As coherence decays (dephasing), C_q → C_μ

Dephasing γ     C(γ)       Advantage   
----------------------------------------
0.0             0.2502     0.7498       ← pure quantum
0.2             0.5667     0.4333      
0.4             0.7693     0.2307      
0.6             0.9008     0.0992      
0.8             0.9756     0.0244      
1.0             1.0000     0.0000       ← fully classical

→ Decoherence destroys off-diagonals → entropy increases → advantage vanishes


In [12]:
# Question 3: How does estimation error propagate?
print("=== Error Propagation (Simulation) ===\n")
print("If we estimate signal states with noise, how wrong is Ĉ_q?\n")

np.random.seed(42)


def add_noise_to_signal_state(s, noise_level):
    """Add noise to signal state amplitudes and renormalize."""
    s_noisy = s + noise_level * np.random.randn(len(s))
    s_noisy = np.maximum(s_noisy, 0)  # Keep positive
    norm = np.sqrt(np.sum(s_noisy**2))
    if norm > 0:
        s_noisy = s_noisy / norm
    return s_noisy


# True perturbed coin at p=0.3
p = 0.3
s_A_true = np.array([np.sqrt(1 - p), np.sqrt(p)])
s_B_true = np.array([np.sqrt(p), np.sqrt(1 - p)])
pi = np.array([0.5, 0.5])

C_q_true, _ = quantum_complexity_simple([s_A_true, s_B_true], pi)

print(f"True C_q = {C_q_true:.4f} bits\n")
print(f"{'Noise Level':<15} {'Mean Ĉ_q':<12} {'Std Dev':<12} {'Rel. Error'}")
print("-" * 55)

for noise in [0.01, 0.05, 0.1, 0.2, 0.3]:
    C_q_estimates = []
    for _ in range(200):
        s_A_noisy = add_noise_to_signal_state(s_A_true, noise)
        s_B_noisy = add_noise_to_signal_state(s_B_true, noise)
        C_q_est, _ = quantum_complexity_simple([s_A_noisy, s_B_noisy], pi)
        C_q_estimates.append(C_q_est)

    mean_est = np.mean(C_q_estimates)
    std_est = np.std(C_q_estimates)
    rel_error = abs(mean_est - C_q_true) / max(C_q_true, 0.01) * 100

    print(f"{noise:<15.2f} {mean_est:<12.4f} {std_est:<12.4f} {rel_error:<.1f}%")

print("\n→ C_q is reasonably robust to small estimation errors")

=== Error Propagation (Simulation) ===

If we estimate signal states with noise, how wrong is Ĉ_q?

True C_q = 0.2502 bits

Noise Level     Mean Ĉ_q     Std Dev      Rel. Error
-------------------------------------------------------
0.01            0.2501       0.0130       0.0%
0.05            0.2497       0.0615       0.2%
0.10            0.2503       0.1191       0.0%
0.20            0.2838       0.1970       13.4%
0.30            0.3776       0.2862       50.9%

→ C_q is reasonably robust to small estimation errors


## Summary

The deep dive is complete. Key insights:

| Part | Focus | Central Theme |
|------|-------|---------------|
| I | Foundations | Diagonal = classical |
| II | Core Concepts | Off-diagonal = quantum |
| III | Computation | Interference uses off-diagonals |
| IV | Connections | $C_q < C_\mu$ because quantum doesn't waste |

**The One Idea:**

> Density matrices reveal everything. Diagonal is classical. Off-diagonal is quantum. That's the whole story.